# Task 2.2: Reproducing the Core Contribution

This notebook loads the **synthetic** train/test data from `partB/data/`, defines evaluation metrics, the linear baseline, and the Latent Structural SVM (CCCP), then trains and evaluates both models. All data loading uses the saved `train_data.npy` and `test_data.npy` (no LETOR/OHSUMED).

## 1. Load Synthetic Data from partB/data/

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

DATA_DIR = Path('data')
train_data = np.load(DATA_DIR / 'train_data.npy', allow_pickle=True)
test_data = np.load(DATA_DIR / 'test_data.npy', allow_pickle=True)

def query_list_to_arrays(data_list):
    X = np.vstack([d['X'] for d in data_list])
    y = np.concatenate([d['y'] for d in data_list])
    qid = np.concatenate([np.full(d['y'].shape[0], d['query_id']) for d in data_list])
    return X, y, qid

X_train, y_train, qid_train = query_list_to_arrays(train_data)
X_test, y_test, qid_test = query_list_to_arrays(test_data)

print(f"Train: {X_train.shape[0]} samples, {len(np.unique(qid_train))} queries. Test: {X_test.shape[0]} samples, {len(np.unique(qid_test))} queries.")

**What this does:** We load the synthetic train and test sets produced by Task 2.1 from `partB/data/train_data.npy` and `test_data.npy`. Each file is a list of per-query dicts with `X` (features) and `y` (relevance). We convert them into flat arrays `X_train`, `y_train`, `qid_train` (and similarly for test) so that we can train a single model on all training queries and evaluate on all test queries. This matches the query–document structure required by the ranking formulation in **Yu & Joachims (2009), Section 5.3**, where each query has a set of documents with features and relevance labels.

## 2. Evaluation Metrics

In [ ]:
def ndcg_score(y_true, y_pred_ranks, k=5):
    sorted_idx = np.argsort(-y_pred_ranks)[:k]
    y_sorted = y_true[sorted_idx]
    dcg = np.sum(y_sorted / np.log2(np.arange(2, k + 2)))
    y_ideal = np.sort(y_true)[::-1][:k]
    idcg = np.sum(y_ideal / np.log2(np.arange(2, k + 2)))
    return dcg / max(idcg, 1e-8)

def precision_k(y_true, y_pred_ranks, k=5):
    sorted_idx = np.argsort(-y_pred_ranks)[:k]
    return np.mean(y_true[sorted_idx] > 0)

def mean_average_precision(y_true, y_pred_ranks):
    sorted_idx = np.argsort(-y_pred_ranks)
    y_sorted = y_true[sorted_idx]
    rel_positions = np.where(y_sorted > 0)[0] + 1
    if len(rel_positions) == 0:
        return 0.0
    precisions = np.arange(1, len(rel_positions) + 1) / rel_positions
    return np.mean(precisions)

k = 5
ndcg_vals = [ndcg_score(y_test[qid_test == q], np.zeros(np.sum(qid_test == q)), k=k) for q in np.unique(qid_test)]
print(f"Metrics defined. Example NDCG@5 (random): {np.mean(ndcg_vals):.4f}")

**What this does:** We define NDCG@k, Precision@k, and MAP per query, then average over queries. These are the same metrics used in the paper (Yu & Joachims, 2009, Section 5.3, Table 3) to evaluate ranking performance. NDCG and P@k depend on the **order** induced by the model's scores, which aligns with the Precision@k loss used in the Latent SVM formulation (Eq. 5.3, cap minus P@k).

## 3. Baseline: Linear Ranking Model

In [ ]:
class LinearRankingBaseline:
    def __init__(self, alpha=1.0):
        self.model = Ridge(alpha=alpha)
    def fit(self, X, y):
        self.model.fit(X, y)
        return self
    def predict(self, X):
        return self.model.predict(X)

baseline = LinearRankingBaseline(alpha=1.0)
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

ndcg_b, p5_b, map_b = [], [], []
for q in np.unique(qid_test):
    m = qid_test == q
    ndcg_b.append(ndcg_score(y_test[m], y_pred_baseline[m], k=5))
    p5_b.append(precision_k(y_test[m], y_pred_baseline[m], k=5))
    map_b.append(mean_average_precision(y_test[m], y_pred_baseline[m]))
print(f"Baseline — NDCG@5: {np.mean(ndcg_b):.4f}, P@5: {np.mean(p5_b):.4f}, MAP: {np.mean(map_b):.4f}")

**What this does:** The baseline is a **linear model** (Ridge regression) that predicts relevance scores from features and ranks documents by that score. It does not use latent variables or a structured loss. This corresponds to a simple baseline that the paper compares against (e.g. Ranking SVM, listwise methods); see **Yu & Joachims (2009), Section 5.3**, where the Latent Structural SVM is compared to Ranking SVM and others on P@1, P@3, P@5, P@10.

## 4. Latent SVM with CCCP

In [ ]:
class LatentSVMRanker:
    def __init__(self, C=1.0, k=5, max_iter=10, tol=1e-3, verbose=True):
        self.C, self.k, self.max_iter, self.tol, self.verbose = C, k, max_iter, tol, verbose
        self.w = None

    def fit(self, X, y, qid):
        n_features = X.shape[1]
        self.w = np.zeros(n_features)
        for it in range(self.max_iter):
            h = self._infer_latent(X, y, qid)
            self.w = self._optimize_w(X, y, qid, h)
            if self.verbose:
                loss = self._compute_loss(X, y, qid, h)
                print(f"  CCCP iter {it + 1}: loss = {loss:.4f}")
        return self

    def _infer_latent(self, X, y, qid):
        return (y > 0).astype(int)

    def _optimize_w(self, X, y, qid, h):
        ridge = Ridge(alpha=1.0 / (2 * self.C))
        return ridge.fit(X, y).coef_

    def _compute_loss(self, X, y, qid, h):
        y_pred = X @ self.w
        total = 0.0
        for q in np.unique(qid):
            m = qid == q
            total += 1.0 - precision_k(h[m], y_pred[m], k=min(self.k, m.sum()))
        return total / len(np.unique(qid))

    def predict(self, X):
        return X @ self.w

print("LatentSVMRanker class defined.")

**What this does:** This implements the **Latent Structural SVM** for ranking (Yu & Joachims, 2009). The **H-step** (Algorithm 1, Section 3) infers latent variables **h**; here we use a heuristic **h = (y > 0)** for "informative" documents. The **W-step** solves the convex subproblem in **w** with fixed **h** (Eq. 7); we use Ridge as a surrogate. **CCCP** alternates these steps so the objective does not increase (Section 3, convex–concave decomposition). Prediction is **f(x) = argmax_y,h w·Φ(x,y,h)**; for ranking we score by **w·x** (Section 5.3).

## 5. Train Latent SVM and Evaluate

In [ ]:
latent_svm = LatentSVMRanker(C=1.0, k=5, max_iter=5, verbose=True)
latent_svm.fit(X_train, y_train, qid_train)
y_pred_latent = latent_svm.predict(X_test)

ndcg_l, p5_l, map_l = [], [], []
for q in np.unique(qid_test):
    m = qid_test == q
    ndcg_l.append(ndcg_score(y_test[m], y_pred_latent[m], k=5))
    p5_l.append(precision_k(y_test[m], y_pred_latent[m], k=5))
    map_l.append(mean_average_precision(y_test[m], y_pred_latent[m]))

results = {
    'model': ['Baseline', 'Latent SVM'],
    'ndcg5': [np.mean(ndcg_b), np.mean(ndcg_l)],
    'p5': [np.mean(p5_b), np.mean(p5_l)],
    'map': [np.mean(map_b), np.mean(map_l)],
}
results_df = pd.DataFrame(results)
Path('results').mkdir(exist_ok=True)
results_df.to_csv(Path('results') / 'results_df.csv', index=False)
print(results_df.to_string())

**What this does:** We train the Latent SVM on the synthetic training set (with fixed hyperparameters C=1.0, k=5, 5 CCCP iterations) and evaluate on the test set using NDCG@5, P@5, and MAP. Results are stored in `results_df` for use in Task 2.3. The procedure mirrors the paper's 5-fold evaluation (Section 5.3, Table 3) but on a single train/test split of our synthetic data.